# 01 — Insurance Claims Dataset Audit

**Project:** Insurance Claim Fraud Detection  
**Purpose:** establish a reproducible baseline before cleaning, feature engineering, or model training.

This notebook does not train a model and does not build the interactive application. It inspects the supplied CSV, checks data quality, documents the fraud-label distribution, and saves audit outputs for the thesis.

## How to run
1. Place this notebook in the root of the cloned repository, next to `insurance_claims.csv`.
2. Install dependencies if needed: `pip install pandas numpy matplotlib seaborn jupyter`.
3. Run every cell from top to bottom.
4. Keep the generated `reports/dataset_audit/` files for the EDA and thesis chapters.

**Important:** the repository CSV has an extra `_c39` column and uses `?` as a missing-value marker in some fields. This audit reports these issues rather than silently hiding them.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

DATA_PATH = Path("insurance_claims.csv")
OUTPUT_DIR = Path("reports") / "dataset_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Put this notebook beside insurance_claims.csv."
    )

# Convert the dataset's '?' marker to a real missing value (NaN).
df = pd.read_csv(DATA_PATH, na_values=["?"], keep_default_na=True)
df.columns = df.columns.str.strip()

print(f"Dataset loaded: {DATA_PATH.resolve()}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())

## 1. Dataset structure and target definition

The prediction target is `fraud_reported`: `Y` means the dataset labels the claim as fraudulent, and `N` means it labels the claim as non-fraudulent. We preserve the original label in the raw data and create a numeric target only inside the later modeling workflow.

In [ ]:
TARGET = "fraud_reported"

if TARGET not in df.columns:
    raise KeyError(f"Expected target column {TARGET!r} was not found.")

structure = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "non_null_count": df.notna().sum().values,
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().mean().values * 100).round(2),
    "unique_values_including_na": [df[col].nunique(dropna=False) for col in df.columns],
})

display(structure)
structure.to_csv(OUTPUT_DIR / "column_structure.csv", index=False)
print("Target labels found:", sorted(df[TARGET].dropna().astype(str).unique().tolist()))
print("Target missing values:", int(df[TARGET].isna().sum()))

## 2. Data-quality checks

We check duplicated records, missing values, columns containing only missing values, unexpected target labels, and repeated identifier-like fields. A missing marker is not automatically a reason to delete a row; the treatment will be decided during preprocessing.

In [ ]:
all_missing_columns = df.columns[df.isna().all()].tolist()
missing_by_column = (
    df.isna().sum().rename("missing_count").to_frame()
      .assign(missing_percent=lambda x: (x["missing_count"] / len(df) * 100).round(2))
      .query("missing_count > 0")
      .sort_values("missing_count", ascending=False)
)

quality_checks = {
    "row_count": int(len(df)),
    "column_count": int(df.shape[1]),
    "duplicate_rows": int(df.duplicated().sum()),
    "all_missing_columns": all_missing_columns,
    "total_missing_cells": int(df.isna().sum().sum()),
    "target_missing": int(df[TARGET].isna().sum()),
    "target_labels": sorted(df[TARGET].dropna().astype(str).unique().tolist()),
}
print(json.dumps(quality_checks, indent=2))
display(missing_by_column)
missing_by_column.to_csv(OUTPUT_DIR / "missing_values_by_column.csv")

identifier_candidates = [c for c in ["policy_number", "insured_zip", "incident_location"] if c in df.columns]
identifier_summary = pd.DataFrame({
    "column": identifier_candidates,
    "unique_values": [df[c].nunique(dropna=True) for c in identifier_candidates],
    "unique_ratio": [round(df[c].nunique(dropna=True) / len(df), 3) for c in identifier_candidates],
})
display(identifier_summary)

## 3. Fraud-label distribution

Accuracy can be misleading when the classes are not equally represented. Record counts and percentages for both classes. Later model evaluation will emphasize fraud-class precision, recall, F1-score, PR-AUC, ROC-AUC, and the confusion matrix—not accuracy alone.

In [ ]:
target_counts = df[TARGET].value_counts(dropna=False).rename_axis("fraud_reported").reset_index(name="count")
target_counts["percentage"] = (target_counts["count"] / len(df) * 100).round(2)
display(target_counts)
target_counts.to_csv(OUTPUT_DIR / "target_distribution.csv", index=False)

plt.figure(figsize=(7, 5))
sns.barplot(data=target_counts, x="fraud_reported", y="count")
plt.title("Insurance claim labels in the supplied dataset")
plt.xlabel("fraud_reported (Y = labelled fraud, N = labelled non-fraud)")
plt.ylabel("Number of claims")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "target_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Numeric and categorical feature overview

This section is descriptive only. Do not interpret a group with a higher fraud rate as proof of fraud or as evidence of causation. Small group sizes can produce unstable percentages.

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()
print(f"Numeric columns ({len(numeric_columns)}): {numeric_columns}")
print(f"Non-numeric columns ({len(categorical_columns)}): {categorical_columns}")

if numeric_columns:
    numeric_summary = df[numeric_columns].describe().T
    display(numeric_summary)
    numeric_summary.to_csv(OUTPUT_DIR / "numeric_summary.csv")

categorical_summary = pd.DataFrame({
    "column": categorical_columns,
    "unique_non_missing": [df[c].nunique(dropna=True) for c in categorical_columns],
    "missing_count": [df[c].isna().sum() for c in categorical_columns],
    "most_frequent_value": [df[c].mode(dropna=True).iloc[0] if not df[c].mode(dropna=True).empty else np.nan for c in categorical_columns],
    "most_frequent_count": [df[c].value_counts(dropna=True).iloc[0] if not df[c].value_counts(dropna=True).empty else 0 for c in categorical_columns],
})
display(categorical_summary)
categorical_summary.to_csv(OUTPUT_DIR / "categorical_summary.csv", index=False)

## 5. Initial consistency checks

The three claim-component columns are compared with `total_claim_amount`. We quantify the difference rather than assuming the records are perfectly consistent. Date columns are parsed to identify invalid values.

In [ ]:
claim_parts = ["injury_claim", "property_claim", "vehicle_claim"]
if all(c in df.columns for c in claim_parts + ["total_claim_amount"]):
    component_sum = df[claim_parts].sum(axis=1, min_count=len(claim_parts))
    difference = df["total_claim_amount"] - component_sum
    print("Rows with all claim components available:", int(difference.notna().sum()))
    print("Rows where total differs from component sum:", int((difference.dropna() != 0).sum()))
    print("Largest absolute difference:", float(difference.abs().max()))
    display(pd.DataFrame({"total_claim_amount": df["total_claim_amount"], "component_sum": component_sum, "difference": difference}).head(10))

for date_col in ["policy_bind_date", "incident_date"]:
    if date_col in df.columns:
        parsed = pd.to_datetime(df[date_col], errors="coerce")
        print(f"{date_col}: invalid/unparsed dates = {int(parsed.isna().sum())}")

for col in ["age", "months_as_customer", "incident_hour_of_the_day", "number_of_vehicles_involved", "bodily_injuries", "witnesses", "auto_year"]:
    if col in df.columns:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")

## 6. Fraud-rate summaries for selected categorical fields

These are exploratory summaries only. Group sizes are shown alongside fraud rates so tiny categories are not mistaken for strong patterns.

In [ ]:
selected_categorical = [
    c for c in ["incident_type", "incident_severity", "collision_type", "authorities_contacted",
                "property_damage", "police_report_available", "insured_sex", "policy_state"]
    if c in df.columns
]

for col in selected_categorical:
    tmp = df[[col, TARGET]].dropna().copy()
    tmp["fraud_binary"] = tmp[TARGET].map({"Y": 1, "N": 0})
    summary = (tmp.groupby(col, dropna=False)
                  .agg(claim_count=(TARGET, "size"), fraud_count=("fraud_binary", "sum"), fraud_rate=("fraud_binary", "mean"))
                  .assign(fraud_rate_percent=lambda x: (x["fraud_rate"] * 100).round(2))
                  .drop(columns="fraud_rate")
                  .sort_values("claim_count", ascending=False))
    print(f"\n--- {col} ---")
    display(summary)
    summary.to_csv(OUTPUT_DIR / f"fraud_rate_by_{col}.csv")

## 7. Save the audit summary

Review the generated tables and retain the JSON file as a reproducible snapshot. Do not alter the original CSV; cleaning will happen in a separate preprocessing notebook.

In [ ]:
audit_summary = {
    **quality_checks,
    "target_distribution": {
        str(row["fraud_reported"]): {"count": int(row["count"]), "percentage": float(row["percentage"])}
        for _, row in target_counts.iterrows()
    },
    "numeric_columns": numeric_columns,
    "categorical_columns": categorical_columns,
    "identifier_candidates": identifier_summary.to_dict(orient="records"),
}

with open(OUTPUT_DIR / "dataset_audit_summary.json", "w", encoding="utf-8") as f:
    json.dump(audit_summary, f, indent=2, default=str)

print(f"Audit outputs saved in: {OUTPUT_DIR.resolve()}")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)

## 8. Research notes to carry into the thesis

Complete these notes using the actual output:

- Number of claim records: **[fill from output]**
- Number of original CSV columns: **[fill from output]**
- Fraud-labelled records (`Y`): **[fill from output]**
- Non-fraud-labelled records (`N`): **[fill from output]**
- Missing-value markers / missing fields: **[fill from output]**
- Empty or redundant columns: **[fill from output]**
- Duplicate rows: **[fill from output]**
- Important consistency issues: **[fill from output]**

### Preliminary limitations to acknowledge

1. This is a relatively small, publicly circulated sample dataset; results should not be generalized to all insurers or current claim populations.
2. The target represents the labels supplied with the dataset. It is not independent proof of fraud.
3. The dataset includes personal and location-like fields. Identifiers and granular location data need careful justification or exclusion.
4. Some claim and incident fields may only be available after parts of the claim investigation. The intended prediction time must be defined before deciding whether to use them.
5. Model results must be evaluated on a held-out, stratified test set, with all preprocessing learned only from training data.

**End of Step 1.** Do not begin model training yet. Save the audit outputs and share the printed summary/results so we can design the cleaning and preprocessing strategy based on evidence.